# GTSRB 最终测试：12个已训练模型，3种固定条件

本笔记本只推理，不训练，不修改 best.pt，不根据测试结果重新选择模型。

准备：
1. 在 Kaggle 新建笔记本并导入本文件。可选 GPU；CPU 也可运行但较慢。
2. Add Input 添加**训练全部完成后下载的 GTSRB_after_migration.zip 对应的数据集**。之前的 GTSRB Checkpoints Transfer 若仍是39轮旧版本不能用于最终测试。若原续训会话尚在，同一会话已有 /kaggle/working/study，也可直接复用。
3. Add Input 添加原始图片数据集 https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign 。无需通过代码联网下载数据。
4. 第2格必须显示12组均80轮且权重校验通过；之后可 Run All。
5. 完成后下载 GTSRB_Final_Results.zip 并发回。该ZIP包含测试指标、预测、混淆矩阵和场景图，不包含原始图片或训练权重；请保留原始训练备份。

协议：种子42/10/20，各组以验证准确率选择best.pt。统一官方测试清单12,630张，输入32×32，归一化(x/255−0.5)/0.5。clean为原始条件；dark为缩放后亮度×0.5；blur为缩放后GaussianBlur(radius=1)。两种扰动均是合成场景，不能代表真实夜间或实际运动模糊性能。

每完成一个模型的一种条件就保存结果；同一会话重新运行会校验后跳过完整结果。会话重置前仍须下载备份，尚未备份的工作目录不保证保留。


In [ ]:
# 1. 写入锁定的模型与评估代码
import os,sys,json,gzip,base64,subprocess,shutil,zipfile,csv,hashlib
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
PROJECT=Path('/kaggle/working/gtsrb_final_project');PROJECT.mkdir(exist_ok=True)
FILES=json.loads(gzip.decompress(base64.b64decode('H4sIANhv7GoC/8V9i3LbOLLor2AztZdUQlOiJL/ow1PXcZxsajNJys7MVl1HxaJISuJYfCxJ2XJc/vfT3XgQpCjHM+fWOcnElgigATT63Q3O46s0j+J1ZRcPr1z26jv9vfj4/vzqoKof1jG7iqvPcX0wHrlswqo6WMYV28LHt0GVhG/XeXhbWSwv6iTPDs5ZtcrLOtzUlf09O4+Cog6wwWXTCcs3dbGpWbgOqiqGMVcf3rIkw0dlXCU/4ojVOZuMt5MxjH13DU+LdRDCbPO8XrHJdsLCPLvL1xuEWMFQFt/F5QONjjbBms1xMWewxjhlmyxcBdkyjgDW9aXLgkUdl6yKAUTE3n622Dxe5GXcDA6iKEHANvucs1USRTHA39ZlAL2HV/Gn33DCd9cAjqPoe5akBewVFl2Gq+/ZosxT/pmJhizTn9pZJhsWsDicCmet2HuAmP1+fvXx/PO3a+Yx05gHVbxOstiwmBFV+LMSn334MMD+hET2hbB+bmaZ/WsebdbxwP2eMfgTxQsG27sPysgEWAuLbWUT/tnCNNsb12L4nzumH7OmuYzrTZmx93YRRObWYubIYuq/rV2tgiK+cWZsOGTjnQcDbX3XcRGUwXwd4wqv439v4qxOgrW+St9PsqT2fbHMMIF/uQVnWCZRrK+52hRxaQ5sNQBAXgA9jCOTBsG/iRxnMcdiyzLfFJWHDfMkqLz3wbqKB1YDcedPB2JOYLSx+s4u+5He3c6zOxg0bSs4kTTYmjBhiGg80doQlF3k+Rr6wKTEVcldfH63/AoPYbVOt/Mi5F0blOvYstgKdjawsAcStvksUnZwA4NDNf46WaZ5EpmDwYsIT1DWlr2WCzXV7sytjmASLC/DcYtkLBaJ44InsTi4Fx4CAkcRYwZw7DBcH6etvqFq2Y8lC5iWgRSNNUTxViTKLiF1jgvndCz+eZ45cHa0ig4vEL7fBnW4+pyXKR5k3gdorACNFaBcUPNLYAR1jSSTZzAaqBw64O5gY3J3HyNsrx/M7kgp+jntNd1oPO2BeR5zOCApul5GNz8A5nu7jNcbU2LJbFCHlDNod25vRQ4aN4PG5o/WICXzaJIf7E17T23i5CpxPHoRfd4FZRJktaeLdaEDvenkpbQJOBSAWJbXqIukyugSaZAAen8P1pv4sizz0hTDdDqvkEYASxIiQDOFphE6xmq3dfRP+9hR3e4XN0D/zhGxgdNlgxeIHdYlWOdIF1zaWubCEgkTWMzNDGdtGoG0uOmCjECWQ7ZJ4zKoY9Ok1QHXHE0HXYbHYQl2L9GUMCfddsKBoGw25nQOk7D/ZCMWgJ2RIMGPOME7u0P5ku2gKOIsMrnE25VmeFb6RuUf2miYd06Dw9w5j9f8eZ9Wsdo6o0+9EMY/AekGpXk0VeQ7aCM4RUwRrJR4ojK7+AIEJVWSAZKyMDZTqxGWfZiFRuQD+zZI0iRb+hlQQLD2zdS+j5PlqrYYWq6esQgyH2xLoNAsz9a0SpA7noGcbHQQBwzYswaNwJ5bSJ7FVTP94Ew1/IjLnFqQuP+UJtzVgtopmoq/SMLZizVJNDgQkkU4Q1HmiwQ0EdnwcJJgRnuTsZxIGPOA4SE/Pfbr+UXFAEsPZ/AzZulmXSfF+uEgCMMNfAGGYCiioRszUc6M2ftPX75WA5sDvNyG600EJjla0AEYsXdk3gON4trhkCxlR8Oz60u2iAPYp5ooCam/rS+Pf06DEEaAyR4B4SAHj4CFb2YNLld5fovHBR6Dha6EjsyX0xVOA6DZGw9h2CgF1qChXjM4vCTz0WfIgF3RBEttbkFia2rfxiU0+IhgHN995Gjmcz+N8QN48XpoOQJ7FR+jeIwOu5fJdiZuBLG+iO4qBNqlJEpBCS4ToLvSFxTsE/bxh5RE4BgB6WdLtFtpOfKBOE56Ft+B6BED7hNw4oQnlPvLElyLgcv7mfwxcRFawBNOyPynnBB3v8Ldi8W6bAXLTPO7WM6grcOUqxFNgtseDbTc0hh2Vhku6NvULBTOcYKiQW/TE9RMR1MZaEH4XNT4eIQATJwkqEnyaeHJTXcrsy4YGONXYV7E0Nvgx0R6QzAr51PnOS41nr5nryz2KgrqoOXBf1kskhAkP1sk6Gd++HZ99ZYFgGYQ7dUZd47AwQcVmCwSdL0RYUM4L1YBm9Zs/oCPwlu74+0G5RIwU8XqQVjdqc+roFqtk7n6/keVZ+oL6NAoT9XXTbmGrnCEoKOquvs4RtNFPfyRFCjlhDsd5ut1HIogAO9wkW8yOCoLJUUA2IqSsBa9i6DGNcmeX+GrggtHXzygD54Vve7814+f5LiPKVkP9OsyWyF/iW/vkzVM3XL1N3Wyrmw8Ezn8HXyu4hrF9ttzkIseM1Z1XVTucFglEWw3Cuzodlhs5iAjh/KchlEQxEE4io7D2HEOx/DldH7kRKcLJ3ZOJ8ejofE9O7+6+MfH3y8xdPBo0Dn77/HM/W+CB3xaZmUDFpHOFpPJIjoZBeHh6XyxOJ6EJ+NofBjMp6en8WgSTIwulbZgwll14IXH8TQ+moyPR0fH0WR8dDh1xtP5aBGfxidj5+Tw5/A+fFNriydOfBqeArD5PIyO5yfT+fFi7ASnEfyND5Hcf2HnHD3s13eHLos4ZquDNFkHQ/XtYFlX5XyYRofA5UDwZRzmZcSu/3E+PjxioF6D9RpIPM9tqUqrVQBNJtKLlI/olQuatkWzLs2AcTPqD0xfzo0BUtLCbRtF4WqT3aJUSYBIzHWQzqPAZQugexCAzmg8fY0/wMKaG8YAZdqmiNAspXFt6bWyV/E2SgD1takMgCi/z9Z5EPm5YHizzHOlHPEz7AGJnj8/o0d2ehslJSy9BPOw8r6VGyDmeAsi389v6auaGbFG+njWiGHkxQyEIxhg0SHuTZKgDZtM2xoJjAVcAS1kqEa2NBbaGdjNphXsWo11WgAE6oFo96sNbHVrGiii666BV5RJVpvGO4EWIH+wC5sFL9abaqXvUP6hA23LJBu+0hETx77RoNRJGoO69o5GdOZVGVqcGGCpQAv3ghagS481eb8COIIuXA8Hd4nBJVvgHqzYNh1oCLFFcNREtLTdRDiTrnrXyBhaaQzN6c8farQgBm3K6pjsK/Y3j6Du+pYL42IVg6W6SVmaVCma0S57RPhPyHKonUmhSSplwRJkkn5mfFb02Dl/dfaTBiWYWQ0BmfIQ4DwMm+KzYR1HxmCHovhIRVMMjSf+iHZew1h4CjvjK3B7yEGoHfv/JcV7NLJpbXiwP3qOFab9YQM2axiFls0Orgwpti6uLtgiSNZg2YHI0wirx8lDdqPtApf9sPHTOsFDcvtdZ7F3U2CLgMJ+q3yNZpKdVD44RQG6d36dU6+mdR/M3qCC8VtWBYtYWhQsjdN5XBo9O/ghTwkELpdBXQuYDoWonZ8KP5BWbIakkDRPHw1EGKiLhiENYFV4gIz6RnvKaQoaOMgnAVOixxCr98GLuoMxoNNtNFmMgb4cfGJHYClUplgIOiEY3fJEsPvXj1dXX650nX5/fw9u63IJxBPm6TAokuGdo5TTULLDEMTIffffkDTXwTIGbzc7ANShYD+okmVmdIU+bQRsy3xThrH3/VWwqfPvr3Th771E9PdJfqAkGjY0yIywwcwDtPRIaCFwf6vQGaB2/PBP2jwIBaQWJiggQdSeCXWmpuGrR040OMoMDXpdPnTIco/Ca4MlN2wbxkXboLR/u/pE9ItMHO8GJ8RSPFiKhG4ITu7TMgtDWdlKwG2y4A54G8PDzHyMn2BhsAdECbft2pixDWtHI6GU8TjqOT58IggykRTSntOZoB68F2jL5zQep2jrWT33jJrTVJzr/UkFt0e5vVwg/1wQiyMIduWx0QlpvUD06iJ3+CcF7l+Rq/vkaWsd/UwrpkPUfc7r9+AzRWLWXzmfgj+4CSleI5KnLlOAUMEjjxu6EB3+aRH6aHAmM1yjxQm6l8CluaBBNYMU5rqhoA+C3SPUq7gAtxUclIh9/fwB/C9KWKMRcnH9e2VzV5E/AC6N10Nyc0ENFQVy6V1ccocYHOCixExPbBtPVlvgowzm2wPECA+gzO9VwgifSLOb/IkzemRzqfti63vHybjHGGd8j4EHz4DPcRbmaOJ6xqZeHJzseiD3GCKu7ux34A//qyQXZIHmcLyOkFQr78ZA0BjcJ2TgBwoNGDNY8z0/wxUwL+Yl1APcqkn7bcKQhCmhj4C5MagcR954ND7CVMLaXwgF4I3ssa6gqHfLRWkYxOJPMdinBZ18QF5jEHJvctj2eIfcQxXiUigh7zneUDzEO7sdMUmH0B1tNUchEC8WmMZ14JGwkLi/4ihc6HF8lGSVV+Ul2K8cNqD0xsAtGzOSPvAdhU8DtCOddto7ogmD9CVmcGpToFfBH0haJInlediHmi8wtP8xgh4cFuYu3M6i0Vrny1YHYi/X+dw0Xg9f20WRGoMGoWCMkdFekbE9OR2PTnsdicttEaMpz95//Hz+qYkvgsHERwHdosRijw3EJxbU7FEt4knKJh64JadRhYOU7916BoekBxgp/kewtT0TZwAwxFHRwptup6AQ8VhB4Xq7pCCaafiG5QxuRrMd76RA1UCR+2fEcqFNQFuCCRbGI63HHR1GT8NHmvfJ6PSrbqjP7Ia+zRrDmZjdRTlvopZta6bBQIkBl29aSQOXw31qjhWOxZTx8b959JUnyaaTfsWmzlckj9jItqdjQ+dskhQieyfjDSWFmHns0L6iXyaKFu3Q+Plg7okTpViWdoK38YNGtC0UoeGYLe1qBWYSnAb21LCeNWUZaZKZSHnU48BBgQ8H1Tx6rUu5VmqZsogWW7bzjjTIZSZuGcP27D9gNsoUEi5QNNV4aJ0DlXwpeJtU/uP2RkptmmyrpMIT+z+stxVmlYdZc/9bIvxFMhBtxX4RuJtAFVJqRxT2uJuF15WDvX72X2Igxam49g5D7OUHyQ494tGSrKFcBR9BG0/9gnOJOCadwI1ALi8/fDug+DwN5SqpHclA8lrWxGLO82IT03k74IDqAFePy/ppx/Kn41zWlLL466eIEfd1kqJR4BlnRu+h7lPWsEapqKGN0Pte+O7/Uwf/EknYd/aNXNxz+oI98fhqclKw8mV8NBkpJkMPSAWn0BoNsgc0dTCGWqJ9OGjt75K6ylN/hI5P3Gak2PYGeiVoS2HAOc/hW4CHKksuARwQSV4+SCrAeV5qhUo/iJu4SAQ3pkHSBaxFLqssZhogUQwS4PSNkGERzgczVzOW0dobohZDkE9ciDBhS/IamDQNygdKZnBPwV8HDzkl1KQ/XIAX0sgpLjSbc0D7g0om4wgPGH5heJ47uHHko5imw1PCGnrpsnu3KsUIyWeAfo+3Lp3pHU8X3lp3bXwIdAhsEDIkLjgqnvqgI135rTnIOBFJLZMI8EbQ44zPXJIQH/z/XAZHCnhx5ToofG5U3VEAZNTTOw2yZIHJGxVjw3UL30wd8i0/YbFMWqRcI18bX9Xs6Uk4lXykQeaTL2jheYdSdGrH5JoYzUt6yloM3q4VeSbLrDJF6u6ZCi/ux0iMqCoMiwWbZYoziXpErD2m2ggPzjwOspa4pFoPEc+jsg+eNqY6Ng7GUkV+HErLa5J5ZtVXdeuT+81ad30YsZD7SiqsfU4MxwSwg0AEyiuOSWQRBabVexnXmCNSqEtaNTH5vSzdw4E3yayzdtIWNu1AoUuqjoL7NSJOBWIbi9Iw9oS5epDFpnH14W0rT4AdEqx1QMyZZoN19WkgUrz2VVwFaYEFLvbbj58+fr48v2rraf2cOjrwF/Z+nYA3Aiwwp7JyFCELsHKwiIgFJaGakiTJMqkrFgaZCL+wFIgE5+xEnOTC8xoNSWEbb+BE8zI1D5yR5Yxg5SVfc+y9ZAsKrp7gtt/SGrO4qswkBYuUP+7OaJ9Yjj0e/BTaRZ4B7wNB/SlYErka2XvMiILy1nBfvmT78Kcg5+tNKUECdheU1je1FL/9IdhUVRJkb6Gn6dgjfZlYUs+T/1gH4FNlgZkVNpxv8ABLAUOpfihiDx4t1nlQT8YDLFXJqgK0tTm24NAGsCAYBEJ9yMaHh/ZOsZi5PYB9DO1DqzFKpFawdC5AEZYgu6GS9X3anu+nKNF9GV0vPFnKYZ+Xyw0S7lf8RoEeWHcU+YF4bhpCUVpgXCQhhY1kwBtEuAj+UMioO/DgAHnUsITT7RmYAzF6O1IJn+pHGqDq70mq3SJ0Ah7UGAw2Sdm/O4ZHHS3WbAGzJbB8Fem3ZP5h1iyD+kibziPvv4oRsB4zCmwR2/IatLhNkibgYj2w+Rp0/0AFzlQfjJ1BT+7hYmUP1lBtgM99suxbNT7vky0Y/qt4HSHymOiJ9JzDotZrsDiR0Ll/wCuAQkzXFjmgDezGDO/PcBOu2lfnYzFyI0T62KKSHrAzseIPbGbkWaxVhVmrOsEKPkwZPFd5k1fAk3dJCVoZlKrAs2lc/Pb20/m1/68vV/+8/np+celffPn8/uMHNOTc6ej0yD3BU/jfKtux6MOnHPWfGMEvTcl+siBcv8oDFNZUfFN1NZVRiyrqGV5LunwHvaZjkNfWGINEsOl3H799/PIZB3MTweKizuLiCfr84/z6H5fX3qOwo1wjPnQmcThynOl0fBQdzo+C4+k0np6ezqfjhXN0OjqaLhaHzuF4dHgYRMEkOF4cOc7CmS7mR1NnEbci72TwGaPJ4cQZTSeT0zCaT06c0fF4PA4nx6cnUyc8nkwX09Gh48Qnx+H0yDkdHZ6MRtNpcOKcBpOTyUk0b4EkC881Jo4zCo6i0elhGI0Pw+PoOHBOo8nx5DQKT0+jcRQenZ7GYTCfO/PRYXQ4DcLwxAniaBSNpkfhKZb+fL368u3LxZdPsHuSDwCWW7UGV3VgPpOTZRkiyATEM4FGUOWGCwYZX5XBNT4MRkLBW2j1Ks6Y1IrqFhqA5SXHyY+A+wmGuR2CZGYHbISCGH7InRpKl8Ccj+LsXBjPD5+0FU23R1mxRYAOGsJl5++/XV6JO3FGb4280FUEUFdK4C9GyaZiThvGk1qk4n6fywW+qzTYJukmRS8oibgAwUpD8IceztgiKSuQLUUeUvVlnQBWJTwUU7BfImQ06OnGADyQLCD7FZsS1RxMdSmkGYknwUNo2qH8kiFfMl3k8tClxTM+iEpw0GH+DVlESA2qbAvcfVfWSLWrtXgGZvBcsQvCoGZK5UhzVEu9aDEuq5Nwway8tJul6bvPYu7kjCw9X6TPhkmel82oGcfeTpJHy/GIqNM9lqcOBmfdrE5PUofwGtzFPkp7WBOKRJXb8gRSzzDPXHSSzPDMoJY93hpCgj1VmxK0aBUmifCNGpfsDDSEzP9imrmQ6wlzIAszTOVCwhSNqaDiFlaYWsrAAvo+mmLivfDC1I6SYIl3Lc0BdMYuJhlgykuMw6RC5Q0Do+QuiWDawoJxWB0MdjTmv6GJFydPJwPrfhWXsac6/OeoqZUDou4F4/wMjNOAWTgaiPFrtb7XHL6lHrwRD/ZANrsd9ZWKamjJ5YZLKIEV02oGQ7EsjICmQVjm/sKRfRaOjW5Juyha9OITyZ78m+rdyGg0XpsZNChFXPokuQ335lHFXmHkpkA1K0eCY6iWp3YpZ1UPoBe09y2Jt+hbwidP3es+gMvZUxMNuAhAfEYvjAnwtLNFWwA2J3dDPKPPvMFrddvnVXedagH0Gbe65VUDf3jKEdEWAnu2fu6TDHs9kR1PxwTwwjkZWHhK2koRuVJaCxUTm1W9iR4svVaV7mhZpGS4jaOqSUUwFSU9DRtK72C4MB4rEWLyvJW1MP4FhuWSwUMVjhG5MZAuJaguT4n6FqRhE38/02O3YhQA14K3ohDCuymlv9WE5MSAdjWG6jdoYteznTSOAAub+JV/AtWXLKl6X5AJi7echVz2KHrfuJOZTP+Ft6j9YI+PTw1G7/ZcFSRsg/rGZtLf3WuEm8wTKIKPHNV3T371iIP0hCMVmCZYsiQxC/2HBj7KyweBU7HNGxHKJGuiFcrE7jNPqCriPsc6cYDBF8ZH0IC4azAaADJsHX5S+rW7inCx9EjVoNdVmXwdYJctkqUMIWqln4PeXLkJQG6kJQOOIH0lf1N+RoNKfKZtVLAPzzPvLIo0g5F5MuqAnmNBF2YRcZPWbfzg8VQ0K6VUwhnXvpLGs+7iwluPcxwv+sN9IVC7ANM3DQofy9lFXLHYGBa/n1b5eH+kr9ZZ7DW8VSfBCwAQZnM4vWOCOajgW77eNAZaD2H/N40imR3wLXFQ3V39hxMfOOO9i1ExZU7HuKzHyuXC4KbiXFbtBpFnT3sh8tPHFaoj9bw7ira1m/kRex7+7tSnopHqqRu/d4Mzfj+IahDR8419itbTBrAFqAMvb4Z1H+olj95IIp55KNZaR9rW0kKiWHKkVnLTlIBKOSWYmMsL7zF0bzjWQsRa411q8iGxynZeugOqL/baEWoy5Jqk3m68VYVXgTNAv1gviqdy/SS8KFW+oEfTlOLaAOGeDHoGk8+1O/aFwcIXTMAd8p7FvSx0+NwMC2Ymb5zB3yej0cjzRq4sN/3Ko0WRWAPIQuj2NHzUVdVTT0mpNPhCN8MClSC8NQMufEMroMsp/EaPULhk9PxfLnGSbAHWJGDFR+KWDpMKSoF/K+6eCh0lyTWK75IwtuZ4JYCuSFr3ORZ6K8ISFpDC283eNJcAOesa+MIAWntNcMbUjbS2DaYtxdNWBXTvi5V54rclikGEa1JQcVUKyszjm0IMlnWFXGEa4SYKjCblBP6wdzM7Q8GCv9d5VXkj+0y/DnlGwz0MlNlg8C545g9dsYYrt9YD7jxa62VIOaYHPH5jcmvXucmXAy4TTvPGEza8eNmM3bxlxgbbvKp8xCMGmDkg60EDYcHCNzyGCcZ22qpCKMp87vExdpUvaqqHGZyB7wcNcHhL/kAfAFiQ5SuFDSoJXK8659WzA8KNakUQfLyNJAU02Omu+3rzJCNcmfxIX08nbxT50KQDK00yYIZlvfKmk9evxyR+8O00YMxbYM8LA44rLq9xKc/EIyA/wBQoAvw11NmqNRKUpIq1+vydQjio50gP6LBlueM9EIcsPVDy0wJltolFjYkW0EfXBjULprGp9Iycsa0vzkA6L/8WjktpFda/qVY5KeSqLcKKhRiXBqcQBWIj4KBSIEIqFdCGeN/YpCj0eu1jW4ezGx1Dkc80qIt1XtOtT36dQ363N+BEGOdLVTG8M8IuHvAT6o9iLZAEH+wy/IqXbyt5F+7RWORZbS+CNFmDr2q8i/8Ift+wa/BUwASgNh7Qc0aWEWyBiCpgWvhV54Xhci5uPSexL1pkxUaYr/MS0wK/jKeHR+dHAPkX5+hkenyMn945J+djBz+dnELrxJidMVlBq16QBc3vruHH9SV9Ym/AsjZmWlU/oHVoCBwbZ1Sjz6sw+uouZBDGhAEWlRu26jCXNjbCb6qEHprY441hY20E6JYi8cajkTWf51vwS0P0Coyadj042z82WsDY3jF4LOEa3UIYJfMhS/CBtGuBTTqtZWx0nI67595dQhxSNqJf0iCqxLLHiCtvmsArPlVfOvVDsNSm9EdCce8sbbSrPloGMjVFNic94Vdl7PoY2fCLsMYCMpkB4eGOUjeJXzujUbMlCrE9C5bHScC2jPyiaMGGR/HdXwIuozjPr1nFev402Bes+VngkgtbhUJDiu779aqMY34ivF4ITlPqSmANZGwP6bPazFGc4CsExha0kKo3nbE1BYNOU67B1sINW1Vk1Um9jnntDILBYrie47X2nY1FlZAqWA7Mw8E4M6sH49Y+dAkwv2LrwXvHGMzcbvXoXdtIl/wz6GWgDB1cnYuWyxcz0KDPX+hw05ZbbeKdOG/M5MABDL+2neMO1235tat5UJpb66a8QVTM2ic/sx6gD8ZSqminaZHWnpEfGBYJZ4+LaIxbkZL0SADj1zAo6LCnmhUCk1dx7W+BCG8rUy7WajY1OONdTCICj35aDxyy8TUuQ1B4zPz7wKCOyzKJzGBdrALPHtOTdbxEedJQIhBgQVDM5l4YFdfCJPTuRKRgi4o62PfNaDR3GKcDdv3ujKG/MkSfgoGFz6qHrF7FsHYhr0kOi0o4NCGlXuAc0iSfOLUBXUml+wv7ksWwOswUU7ynWuX3GecDlYKxVA6Gqgcx9hTw3iDMa53E7ef4btziu0PLmXTY7q5hOLRSSCXjS20slTCiI9Wpmlt+6OnWQNQoFEQgajoekn84ROtmgwFfHyyLMtlyGaGVpVp61Q9m9LwwHQJQkfaSoWjHuo3jIkpSXgw5sJyW0+fBkUMbYM9EGNYdmJreCH4FW88BOyooPOMtWrBGh7BgvfThid+JYdOxYW0lmUkTj4nrN5L+cAXi2U9oegR2rQXyTUzqP/Q2Ig0R+yAvgv8Mi4bzUJdy7NH0sJeQLxDFnAQUnhnhOcTXiJBulklS2IVZ4YW5htoGPyNfOkK/OUIc1NwK4GTWJjKdwDqC/UViEuxzZbWoQOi+KCiYya2QokZ/j9WTIEBhS/fFGM9gNml4YLh4V/vB4Bst7zGbtY8b99xcrLCQalFXwwBATAK+Jbg8dIHaM2zDWt8DKXZlY1t0StuNU6ikQypt7tIg6SOGBaFcdGFVCUx8wNMh7x0SjpyjPY1QXH7D4+CeSpcdBmdd0nU8giTEoGKSNjHj5ZHnpe1LyEmhE62NZy2FiXWiSaypdaST0x9NKuhksKORwzapaXqlewsLLYLE+mMmxYcIIIWzmz9muDfZjAdqYjmSoT9FHHFmBMILn75/z/h9l0fhjwGQm++v6Nn3VzOM/oA/RBs67eVoXjTEs/onQj/JrAKhjw3ZvKlLQEUVHWNZwlCVGrC5XmvwMxZXqkxXUzKTYWj47miH/TqBrByVbujTCcJ1xql4BkK7Fo4WCRhJZmKVg/4QKFmkwmIi33z2Nw/ddeWPz2ZdeDbeLTK12H4RJDK8jx/BJLxpee8zjLkAQ1RxN0y8j1KnDaWeWMfdC0YBXetRGtUFltHoqdvX4rvf0cN8MzeuM57tknGHfmV4FoWKZGZJqKS+HksgTMQf0iUQ0tg5HVNwiLcodGIzkLXADHuUSZHvrzSEQaeBa08WfQS+Q+S7KtdlmwILe5wxWwGhYkGJnI+2cUCMwLf/E4JeGLybTyTX0lciuAL2tgyoYM2lqWonmqQnZe2pkCKw6enAEt+o1kJ+wauvZ3sua+x5awNA2BellIlUHiLEZ5iPVO8rMAeWcfHbu3P9HQZnooCQffj6G6M3OeBL/AC3DwzvnBwc8MkY5p3EdTICjsSAoVX03VBzghDnDWmQbYI14c3UCjZ54zwIb0HKV7i2LLPncRauUL3xSOxZb6cI39EGthi5m4QL0Q/W57ca/WC9zMG/XKWVqeGsm2Dx+nLUMr6b1zmoUkpQeZ1aI63KRdapWSgVfLzpxw/JpmqeuPeFO029DXm+cipuVVgSYn9nUVGJta6i/6NRPIBFnxmuLNO0+QMfZQ6+EhcOm/BkuBxdvmzxfcugpL6B2QL9sVapwY/dcCW1WcYSKMDVKGsJFMDbqAqZvwb4GeLk13g+51msz9NE6nEuLW5viGA9PpZxe3lhSF5JeczwZQG8CMynvLvvy4vPw4wL/4znE3eqbNFbl+/Fpwo+er8e3lRpXS8xQKBoNbXkU3PzVyYzK1XzENGr5dXF5vsE/C9MA/YkbEQCrzfHdyaDYq3g21/N8+PZeO3c/lk73dyqAsDu/73k8383nyq617kpiWmwe1sx3BuC1N9LRvFY2lGoB2OfkbJn4tUqQ6Nl9PdeVkyLfS802+Nd0ItP9tUndJLbMBQ5DJhH1EYY/BIhdyekmBKMMPO8luRqeuoFoaqvylFTfvpny7jZuQoGIETOXtzuauaTJWCzVkHNHuCYflFWHsf6jl03eBZQO3jAQfy5cMGe1ymo6rVnpxci4oquZ8qDirQ6fdcg49YKrb3vpeu5StwhIZnD8fYmZcHVUHlZJbBb0lTJ0MHeaZpkjCI5lzikW4qth9R7rX94/jd5A+6ZV4C34vT8NieV2JD08XmViqvVsFg91Oy2afm52XYuNLZp2NrhKrfFU097MNcEtPspmNcBn2FeHnT6z8gUjppion+PXkayO7NLz5iD6wYfuJXLDYu0wObBc5Qtb/Ey8EtdFSL0HhHs91fyO1juGPIg4/3vlopyy24yJt7qZjzPEFIDqpzOrYvAbmf6PVNJPoJ4dMJsAvpaUF59FCWjFs8FN3WrLcrrozZrVxR2yQIOjWvA/ekOPA80C3iqQ+wUtP5y6b0kPdtrIBJgGLlZ11VjIQr6NVxlrxpicrCqJHhDS7/A4+XyqX+Kiy+/fv10+e1SgVcCz28EHqbUjixuS4CTabWuTUwUqgkHzSh1q+NlLCjE7oUSuBjonhz1XZSqbCbfFMoEeuT/GAjoCq00EF5gP1XIiV1DrXXHzvP+1A27ngtqaGaBOIj/vUlARgh7Y7cfXaD7eTe6Tdfute9qnLDlm0tvZJCjCbPXURR37tHg65ucNMsBv7+ye03v8KhvjLTnewbI4gv0pdtX8PCO3NN/Afs5iTrCagAA')))
for name,content in FILES.items(): (PROJECT/name).write_text(content)
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('当前设备：',torch.cuda.get_device_name() if DEVICE=='cuda' else 'CPU')
print('本笔记本不执行训练，只评估固定的12个最佳权重。')


In [ ]:
# 2. 查找完整训练结果；明确拒绝旧的39轮迁移包
INPUT=Path('/kaggle/input')
def complete(p):
    try:
        for v in ['baseline','ds','se','ds_se']:
            for s in [42,10,20]:
                r=p/'runs'/f'{v}_s{s}'
                with open(r/'history.csv') as f: rows=list(csv.DictReader(f))
                if [int(q['epoch']) for q in rows]!=list(range(1,81)) or not (r/'best.pt').is_file(): return False
        return (p/'splits/test.csv').is_file()
    except (OSError,ValueError,KeyError): return False
existing=Path('/kaggle/working/study')
if complete(existing):
    STUDY=existing
else:
    candidates=[p.parent.parent for p in INPUT.rglob('split_summary.json') if p.parent.name=='splits' and complete(p.parent.parent)]
    if len(candidates)==1:
        STUDY=candidates[0] # Kaggle Input只读，评估结果另存working
    elif len(candidates)>1:
        raise RuntimeError('发现多个完整study，请只添加一个最终训练结果数据集：'+str(candidates))
    else:
        archives=list(INPUT.rglob('GTSRB_after_migration.zip'))
        assert len(archives)==1,'没有发现最终训练结果。请Add Input添加GTSRB_after_migration.zip对应的数据集；不要使用39轮旧迁移包。'
        extract=Path('/kaggle/working/gtsrb_final_checkpoint_input');extract.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            assert z.testzip() is None,'ZIP损坏'
            for n in z.namelist(): assert (extract/n).resolve().is_relative_to(extract.resolve()),'ZIP含不安全路径'
            z.extractall(extract)
        STUDY=extract/'study'
        assert complete(STUDY),'训练结果不完整，要求12组各80轮'
EXPECTED_WEIGHTS={'baseline_s42': 'a889a64552fbc75443ac50022383281841f019773f512314efa3bedca930410f', 'baseline_s10': 'e7a09f486aae0bbe51b9327ade4befff7090e7fe139c9607b10f947eb7590a8c', 'baseline_s20': '9f796d5b41de51d2eecad179dad81a07f7ce74f71943449dea3364ceb53db149', 'ds_s42': '5a11951ea383b5eb07136313cb15c0d9c5181326cff4c2cb5cc4bb1bd2a28e89', 'ds_s10': '21e7158067e4a055ea22beb217fd59f5958d6844d1ab406c4497f4f89c273ae8', 'ds_s20': 'a7321e61a3697ad931f7ff4dd219e94d5e3fd3757a7707b613e9eac7d69cbaa6', 'se_s42': 'd6cc9e404752e835962664bd935d9c56cc1bddaee8aef5a9cf9188be4cb534ef', 'se_s10': 'd6492c7abf187deef68a7f9f869cadd5a1e0f4f969ba4c20335fd334f9ddecae', 'se_s20': 'b9a5e02d1b52aed76532296bed993fbfe590494b408a363722580391843e3771', 'ds_se_s42': '215244c19fb25d37f24bf5eb9a6625329aad92c06a3db59951a51f05848e742d', 'ds_se_s10': '2f7fded2c257817d6e40abfdabbefa9295dd0762a2cae1a1d6b56e167b4eface', 'ds_se_s20': '447a352218f81140f7e68041d3c89f8d759f85f0760fb67fa951ebd95e5b2bf4'}
for name,digest in EXPECTED_WEIGHTS.items():
    actual=hashlib.sha256((STUDY/'runs'/name/'best.pt').read_bytes()).hexdigest()
    assert actual==digest,f'与本次核验的最佳权重不一致：{name}'
print('12组均80轮，全部best.pt与核验过的文件一致。')
print('结果位置：',STUDY)


In [ ]:
# 3. 查找已挂载的原始图片；不联网下载、不重新划分
with open(STUDY/'splits/test.csv') as f: TEST_ROWS=list(csv.DictReader(f))
candidates=[p.parent for p in INPUT.rglob('Train.csv') if (p.parent/'Test.csv').is_file()]
local=Path('/kaggle/working/gtsrb_data')
if (local/'Train.csv').is_file() and local not in candidates: candidates.append(local)
matches=[p for p in candidates if all((p/r['path']).is_file() for r in TEST_ROWS)]
assert len(matches)==1,'必须挂载一个完整的同版本GTSRB图像数据集；当前可用位置：'+str(matches)
DATA=matches[0]
print('已找到12630张测试图片：',DATA)


In [ ]:
# 4. 统一测试12个模型 × clean/dark/blur，共36次评估；不会训练
OUT=Path('/kaggle/working/GTSRB_final_evaluation')
args=[sys.executable,str(PROJECT/'evaluate_final.py'),'--study',str(STUDY),
      '--data',str(DATA),'--out',str(OUT),'--device',DEVICE,'--batch-size','256','--workers','2']
subprocess.run(args,check=True,cwd=PROJECT)


In [ ]:
# 5. 核对测试数量，导出所有结果与运行源码
assert json.loads((OUT/'COMPLETE.json').read_text())['completed_evaluations']==36
results=json.loads((OUT/'test_results.json').read_text())
for r in results['three_seeds']:
    print(f"{r['variant']:8s} {r['condition']:5s} Accuracy {r['accuracy_mean_pct']:.3f} ± {r['accuracy_sample_std_pp']:.3f}% | Macro-F1 {r['macro_f1_mean_pct']:.3f} ± {r['macro_f1_sample_std_pp']:.3f}%")
archive=Path('/kaggle/working/GTSRB_Final_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,Path('evaluation')/p.relative_to(OUT))
    for p in sorted(PROJECT.glob('*.py')): z.write(p,Path('source')/p.name)
    for p in sorted((STUDY/'splits').glob('*')):
        if p.is_file():z.write(p,Path('splits')/p.name)
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('ZIP存在且校验通过，MiB：',round(archive.stat().st_size/1024**2,2))
os.chdir('/kaggle/working')
from IPython.display import display,FileLink
display(FileLink(archive.name))
print('若链接404，在右侧Output/Files刷新，找到此ZIP后Download。下载成功后再停止会话。')
